In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
!pip install fair-esm

# ESM2 Zeroshot scores

8M --> 0.1116 spearman on train

150M --> 0.3593 spearman on train

650M --> 0.2960 spearman on train

3B --> 0.2608 spearman on train

In [ ]:
"""
ESM-2 650M Zero-Shot Cache Generator
=====================================
Computes masked marginal log-probabilities for all 656 positions
"""

import os
import torch
import numpy as np
import pandas as pd

# ── Config ─────────────────────────────────────────────────────────────────────
DATA_DIR   = os.path.expanduser('~/v_files/GT/sem2/MLB/hackathon/Hackathon_data')
FASTA_PATH = f'{DATA_DIR}/sequence.fasta'
TRAIN_PATH = f'{DATA_DIR}/train.csv'
TEST_PATH  = f'{DATA_DIR}/test.csv'
CACHE_PATH = f'{DATA_DIR}/esm2_3B_zeroshot_scores.npz'

# ── Device ─────────────────────────────────────────────────────────────────────
# Force CPU — MPS has numerical precision issues with ESM-2 log-softmax
# producing near-zero Spearman scores despite correct-looking logit ranges.
# CPU is slower but guaranteed correct.
device = torch.device('cpu')
print("Device: CPU (MPS disabled due to precision issues with ESM-2)")

# ── Load sequence and get all positions ───────────────────────────────────────
with open(FASTA_PATH) as f:
    sequence_wt = f.readlines()[1].strip()
SEQ_LEN = len(sequence_wt)
print(f"Sequence length: {SEQ_LEN}")

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)
all_df   = pd.concat([df_train, df_test], ignore_index=True)
all_positions = sorted(all_df['mutant'].apply(lambda x: int(x[1:-1])).unique())
print(f"Unique positions to score: {len(all_positions)}")

# ── Load ESM-2 650M ────────────────────────────────────────────────────────────
import esm

print("\nLoading ESM-2 3B ...")
model_esm, alphabet = esm.pretrained.esm2_t36_3B_UR50D()
model_esm.eval().to(device)
batch_converter = alphabet.get_batch_converter()
print("Model loaded ✓")

AAs       = list('ACDEFGHIKLMNPQRSTVWY')
aa_tokens = [alphabet.get_idx(aa) for aa in AAs]

# ── Compute masked marginal log-probs ─────────────────────────────────────────
# Batch size 8 is safe for a 656-aa sequence
BATCH_SIZE   = 8
pos_logprobs = {}
positions    = list(all_positions)

print(f"\nScoring {len(positions)} positions (batch size {BATCH_SIZE}) ...")
for i in range(0, len(positions), BATCH_SIZE):
    batch_pos  = positions[i:i+BATCH_SIZE]
    batch_data = [
        (f"p{p}", sequence_wt[:p] + '<mask>' + sequence_wt[p+1:])
        for p in batch_pos
    ]
    _, _, tokens = batch_converter(batch_data)
    tokens = tokens.to(device)

    with torch.no_grad():
        logits = model_esm(tokens, repr_layers=[], return_contacts=False)['logits']

    lp = torch.log_softmax(logits, dim=-1).cpu().float().numpy()

    for j, pos in enumerate(batch_pos):
        pos_logprobs[pos] = {
            aa: float(lp[j, pos + 1, tok])
            for aa, tok in zip(AAs, aa_tokens)
        }

    # Progress
    done = i + len(batch_pos)
    if done % 80 == 0 or done == len(positions):
        print(f"  {done}/{len(positions)} positions scored")

# ── Save cache ─────────────────────────────────────────────────────────────────
np.savez(CACHE_PATH, pos_logprobs=pos_logprobs)
print(f"\nSaved → {CACHE_PATH}")
print(f"Cache size: {os.path.getsize(CACHE_PATH) / 1024:.1f} KB")

# ── Quick sanity check ────────────────────────────────────────────────────────
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge

def zero_shot_scores(df):
    scores = []
    for _, row in df.iterrows():
        m   = row['mutant']
        pos = int(m[1:-1])
        scores.append(pos_logprobs[pos][m[-1]] - pos_logprobs[pos][m[0]])
    return np.array(scores, dtype=np.float32)

zs_train = zero_shot_scores(df_train)
r_zs, _  = spearmanr(df_train['DMS_score'].values, zs_train)
print(f"\nSanity check — zero-shot Spearman on train: {r_zs:.4f}")

Device: CPU (MPS disabled due to precision issues with ESM-2)
Sequence length: 656
Unique positions to score: 656

Loading ESM-2 3B ...


Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t36_3B_UR50D.pt" to /Users/vishnu/.cache/torch/hub/checkpoints/esm2_t36_3B_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t36_3B_UR50D-contact-regression.pt" to /Users/vishnu/.cache/torch/hub/checkpoints/esm2_t36_3B_UR50D-contact-regression.pt


Model loaded ✓

Scoring 656 positions (batch size 8) ...
  80/656 positions scored
  160/656 positions scored
  240/656 positions scored
  320/656 positions scored
  400/656 positions scored
  480/656 positions scored
  560/656 positions scored
  640/656 positions scored
  656/656 positions scored

Saved → /Users/vishnu/v_files/GT/sem2/MLB/hackathon/Hackathon_data/esm2_3B_zeroshot_scores.npz
Cache size: 165.9 KB

Sanity check — zero-shot Spearman on train: 0.2608


In [18]:
import numpy as np
cache = np.load('/Users/vishnu/v_files/GT/sem2/MLB/hackathon/Hackathon_data/esm2_3B_zeroshot_scores.npz', allow_pickle=True)
pos_logprobs = cache['pos_logprobs'].item()

# Check scores for position 0
print("Position 0 logprobs sample:", list(pos_logprobs[0].items())[:5])

# Check if scores are all the same
import pandas as pd
df_train = pd.read_csv('/Users/vishnu/v_files/GT/sem2/MLB/hackathon/Hackathon_data/train.csv')
scores = []
for _, row in df_train.iterrows():
    m = row['mutant']; pos = int(m[1:-1])
    scores.append(pos_logprobs[pos][m[-1]] - pos_logprobs[pos][m[0]])
import numpy as np
scores = np.array(scores)
print(f"Score stats: mean={scores.mean():.4f} std={scores.std():.4f} min={scores.min():.4f} max={scores.max():.4f}")

Position 0 logprobs sample: [('A', -5.798081874847412), ('C', -7.321051597595215), ('D', -7.422849655151367), ('E', -6.76283073425293), ('F', -7.410057067871094)]
Score stats: mean=-11.6202 std=3.7409 min=-21.3607 max=3.1199


# Round 1 - Ridge regression + ESM-2 zero-shot scores  (Greedy selection )



In [ ]:
"""
Protein Fitness Prediction — ESM-2 Zero-Shot + Ridge
=====================================================
Uses ESM-2 150M cache (best model found: train Spearman 0.3593).
Ridge calibrates zero-shot score → DMS scale.

"""

import os
import torch
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

# ── Config — only section you need to edit ─────────────────────────────────────
DATA_DIR   = os.path.expanduser('~/v_files/GT/sem2/MLB/hackathon/Hackathon_data')
FASTA_PATH = f'{DATA_DIR}/sequence.fasta'
TRAIN_PATH = f'{DATA_DIR}/train.csv'
TEST_PATH  = f'{DATA_DIR}/test.csv'
CACHE_PATH = f'{DATA_DIR}/esm2_150M_zeroshot_scores.npz'

QUERY_PATHS = [f'{DATA_DIR}/query1_labeled.csv']   # ← update each round
N_ENSEMBLE  = 5    
BOOTSTRAP   = 0.9
N_QUERY     = 100
ROUND       = 2    # ← update each round


device = torch.device('cpu')
print("Device: CPU")

# ── Load data ──────────────────────────────────────────────────────────────────
with open(FASTA_PATH) as f:
    sequence_wt = f.readlines()[1].strip()
SEQ_LEN = len(sequence_wt)

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)
for qp in QUERY_PATHS:
    df_train = pd.concat([df_train, pd.read_csv(qp)], ignore_index=True)
df_train = df_train.drop_duplicates(subset='mutant').reset_index(drop=True)

print(f"Train: {len(df_train)} | Test: {len(df_test)} | Seq: {SEQ_LEN}")

# ── ESM-2 zero-shot: load cache ────────────────────────────────────────────────
# Run compute_esm2_cache.py first if cache doesn't exist.
all_df        = pd.concat([df_train, df_test], ignore_index=True)
all_positions = all_df['mutant'].apply(lambda x: int(x[1:-1])).unique()
AAs           = list('ACDEFGHIKLMNPQRSTVWY')

if os.path.exists(CACHE_PATH):
    print(f"Loading ESM-2 cache ...")
    pos_logprobs = np.load(CACHE_PATH, allow_pickle=True)['pos_logprobs'].item()
    print(f"  {len(pos_logprobs)} positions loaded")
else:
    print("Cache not found — run compute_esm2_cache.py first.")
    print("Computing with 150M on CPU as fallback ...")
    import esm
    model_esm, alphabet = esm.pretrained.esm2_t30_150M_UR50D()
    model_esm.eval().to(device)
    batch_converter = alphabet.get_batch_converter()
    aa_tokens       = [alphabet.get_idx(aa) for aa in AAs]

    pos_logprobs = {}
    positions    = list(all_positions)
    for i in range(0, len(positions), 8):
        batch_pos  = positions[i:i+8]
        batch_data = [(f"p{p}", sequence_wt[:p]+'<mask>'+sequence_wt[p+1:]) for p in batch_pos]
        _, _, tokens = batch_converter(batch_data)
        with torch.no_grad():
            logits = model_esm(tokens.to(device), repr_layers=[], return_contacts=False)['logits']
        lp = torch.log_softmax(logits, dim=-1).cpu().float().numpy()
        for j, pos in enumerate(batch_pos):
            pos_logprobs[pos] = {aa: float(lp[j, pos+1, tok]) for aa, tok in zip(AAs, aa_tokens)}
        if i % 80 == 0:
            print(f"  {i+len(batch_pos)}/{len(positions)} positions")

    del model_esm
    np.savez(CACHE_PATH, pos_logprobs=pos_logprobs)
    print(f"Saved cache → {CACHE_PATH}")


def zero_shot_scores(df):
    """log P(mutant | context) - log P(wt | context)"""
    scores = []
    for _, row in df.iterrows():
        m   = row['mutant']
        pos = int(m[1:-1])
        scores.append(pos_logprobs[pos][m[-1]] - pos_logprobs[pos][m[0]])
    return np.array(scores, dtype=np.float32)

zs_train = zero_shot_scores(df_train)
zs_test  = zero_shot_scores(df_test)
y_train  = df_train['DMS_score'].values.astype(np.float32)

r_zs, _ = spearmanr(y_train, zs_train)
print(f"Zero-shot Spearman on train: {r_zs:.4f}")


# ── Ridge calibration ──────────────────────────────────────────────────────────
cal   = Ridge(alpha=1.0)
cal.fit(zs_train.reshape(-1, 1), y_train)
y_pred = np.clip(cal.predict(zs_test.reshape(-1, 1)), 0, 1)

# Position-split CV — honest leaderboard estimate
train_pos  = df_train['mutant'].apply(lambda x: int(x[1:-1])).values
unique_pos = np.unique(train_pos)
np.random.seed(42)
np.random.shuffle(unique_pos)
fold_size  = len(unique_pos) // 5
cv_scores  = []
for fold in range(5):
    val_pos  = set(unique_pos[fold*fold_size:(fold+1)*fold_size])
    val_mask = np.array([p in val_pos for p in train_pos])
    c = Ridge(alpha=1.0)
    c.fit(zs_train[~val_mask].reshape(-1, 1), y_train[~val_mask])
    r, _ = spearmanr(y_train[val_mask], c.predict(zs_train[val_mask].reshape(-1, 1)))
    cv_scores.append(r)
print(f"Position-split CV Spearman: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# ── Save submission ────────────────────────────────────────────────────────────
submission = pd.DataFrame({'id': range(len(df_test)), 'DMS_score': y_pred})
submission.to_csv(f'{DATA_DIR}/predictions.csv', index=False)
submission.to_csv(f'{DATA_DIR}/submission_round{ROUND}.csv', index=False)
print(f"Saved predictions.csv -> upload to Kaggle")

# # ── Thompson sampling query selection ─────────────────────────────────────────
# # GBM ensemble on zero-shot score only — no position features.
# X_train = zs_train.reshape(-1, 1)
# X_test  = zs_test.reshape(-1, 1)

# print("\nTraining query selection ensemble ...")
# np.random.seed(42)
# ensemble = []
# for i in range(N_ENSEMBLE):
#     n   = len(X_train)
#     idx = np.random.choice(n, size=int(n * BOOTSTRAP), replace=False)
#     m   = GradientBoostingRegressor(
#         n_estimators=200, learning_rate=0.1,
#         max_depth=3, random_state=i
#     )
#     m.fit(X_train[idx], y_train[idx])
#     ensemble.append(m)

# all_preds = np.stack([m.predict(X_test) for m in ensemble])

# seen_pos            = set(df_train['mutant'].apply(lambda x: int(x[1:-1])))
# df_q                = df_test.copy()
# df_q['pos']         = df_q['mutant'].apply(lambda x: int(x[1:-1]))
# df_q['cal_fitness'] = y_pred
# df_q['std_fitness'] = all_preds.std(axis=0)
# df_q['ts_score']    = all_preds[np.random.randint(N_ENSEMBLE)]

# query_df = (
#     df_q[~df_q['pos'].isin(seen_pos)]
#     .sort_values('ts_score', ascending=False)
#     .drop_duplicates(subset='pos')
#     .head(N_QUERY)
# )
# query_df[['mutant']].to_csv(f'{DATA_DIR}/query_round{ROUND}.csv', index=False)
# print(f"Saved query_round{ROUND}.csv ({len(query_df)} mutations, {query_df['pos'].nunique()} positions)")

# print(f"\nTop 10 predicted mutations:")
# print(df_q.nlargest(10, 'cal_fitness')[['mutant', 'cal_fitness', 'std_fitness']].to_string(index=False))



Device: CPU
Train: 1240 | Test: 11324 | Seq: 656
Loading ESM-2 cache ...
  656 positions loaded
Zero-shot Spearman on train: 0.4912
Position-split CV Spearman: 0.4854 ± 0.0949
Saved predictions.csv → upload to Kaggle

Training query selection ensemble ...
Saved query_round2.csv (100 mutations, 100 positions)

Top 10 predicted mutations:
mutant  cal_fitness  std_fitness
 T243R     0.748029     0.013944
 K198E     0.726361     0.013944
 P571G     0.724324     0.003862
 G595K     0.724314     0.044556
 L589E     0.723151     0.044556
  C13S     0.720186     0.009728
 L589K     0.718387     0.044556
 P571E     0.717726     0.013394
 G595R     0.717549     0.044556
 V574E     0.717194     0.004249

── Round 2 complete ──────────────────────────────────────────────────────
  predictions.csv         → Kaggle upload
  query_round2.csv         → hackathon API

After receiving labels:
  1. Save as /Users/vishnu/v_files/GT/sem2/MLB/hackathon/Hackathon_data/query2_labeled.csv  (columns: mutant, DM

how we selected the top 100 for query:

1. Train 5 GBM models on (zero_shot_score → DMS_score) using 90% of training data each, with different random seeds.
2. Randomly pick 1 model (Thompson sampling) — say model 3 out of 5.
3. That model predicts fitness for all 11,324 test mutants.
4. Filter to unseen positions — remove any mutant whose position already appears in train.csv (the 60 seen positions).
5. Sort by predicted fitness descending, then drop_duplicates(pos) — keeps only the highest-scoring mutant per position, enforcing that you cover 100 different positions rather than 100 mutations at the same few positions.
6. Take top 100.

In [1]:
import pandas as pd, os
DATA_DIR = os.path.expanduser('~/v_files/GT/sem2/MLB/hackathon/Hackathon_data')
df = pd.read_csv(f'{DATA_DIR}/query_round1.csv')
with open(f'{DATA_DIR}/query.txt', 'w') as f:
    f.write('\n'.join(df['mutant'].tolist()))
print("Done — submit query.txt")

Done — submit query.txt


# Gaussian Process model

predictions based on first round (Greedy query/Exploitation)

GP predictions on kaggle - 0.43769

In [3]:
"""
Protein Fitness Prediction — Gaussian Process
==============================================
Input features (all from ESM-2 cache, generalize to unseen positions):
  1. zero_shot_score   = log P(mut) - log P(wt)
  2. position_entropy  = Shannon entropy of ESM distribution at position
  3. wt_log_prob       = log P(wt | context)
  4. normalized_pos    = position / seq_len  (structural context proxy)

Kernel: Matern-5/2 with ARD (learns separate lengthscale per feature)

"""

import os
import torch
import gpytorch
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler

# ── Config ─────────────────────────────────────────────────────────────────────
DATA_DIR   = os.path.expanduser('~/v_files/GT/sem2/MLB/hackathon/Hackathon_data')
TRAIN_PATH = f'{DATA_DIR}/train.csv'
TEST_PATH  = f'{DATA_DIR}/test.csv'
FASTA_PATH = f'{DATA_DIR}/sequence.fasta'
CACHE_PATH = f'{DATA_DIR}/esm2_150M_zeroshot_scores.npz'

QUERY_PATHS = [f'{DATA_DIR}/query1_labeled.csv']  # ← update each round
ROUND       = 2                                    # ← update each round
GP_ITERS    = 300

# ── Load data ──────────────────────────────────────────────────────────────────
with open(FASTA_PATH) as f:
    sequence_wt = f.readlines()[1].strip()
SEQ_LEN = len(sequence_wt)

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)
for qp in QUERY_PATHS:
    df_train = pd.concat([df_train, pd.read_csv(qp)], ignore_index=True)
df_train = df_train.drop_duplicates(subset='mutant').reset_index(drop=True)

print(f"Train: {len(df_train)} | Test: {len(df_test)} | Seq: {SEQ_LEN}")

# ── Load ESM-2 cache ───────────────────────────────────────────────────────────
AAs = list('ACDEFGHIKLMNPQRSTVWY')
print("Loading ESM-2 cache ...")
pos_logprobs = np.load(CACHE_PATH, allow_pickle=True)['pos_logprobs'].item()
print(f"  {len(pos_logprobs)} positions loaded")

# ── Feature extraction ─────────────────────────────────────────────────────────
def build_features(df):
    feats = []
    for _, row in df.iterrows():
        m      = row['mutant']
        wt, mt = m[0], m[-1]
        pos    = int(m[1:-1])
        lp     = pos_logprobs[pos]

        zs       = lp[mt] - lp[wt]
        logprobs = np.array([lp[aa] for aa in AAs])
        probs    = np.exp(logprobs); probs /= probs.sum()
        entropy  = float(-np.sum(probs * np.log(probs + 1e-10)))
        wt_logp  = lp[wt]
        norm_pos = pos / SEQ_LEN

        feats.append([zs, entropy, wt_logp, norm_pos])
    return np.array(feats, dtype=np.float32)

print("Extracting features ...")
X_train_raw = build_features(df_train)
X_test_raw  = build_features(df_test)
y_train     = df_train['DMS_score'].values.astype(np.float32)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

# ── GP Model ───────────────────────────────────────────────────────────────────
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module  = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(
                nu=2.5,
                ard_num_dims=train_x.shape[1]
            )
        )

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x),
            self.covar_module(x)
        )


def train_gp(X, y, n_iter=GP_ITERS, lr=0.1):
    train_x = torch.tensor(X)
    train_y = torch.tensor(y)

    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model      = ExactGPModel(train_x, train_y, likelihood)
    model.train(); likelihood.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(n_iter):
        optimizer.zero_grad()
        loss = -mll(model(train_x), train_y)
        loss.backward()
        optimizer.step()
        if (i + 1) % 50 == 0:
            ls = model.covar_module.base_kernel.lengthscale.detach().numpy().flatten()
            print(f"  Iter {i+1}/{n_iter} | Loss: {loss.item():.4f} | "
                  f"Lengthscales: {np.round(ls, 3)}")

    return model, likelihood


def gp_predict(model, likelihood, X):
    model.eval(); likelihood.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred = likelihood(model(torch.tensor(X)))
    return pred.mean.numpy(), pred.stddev.numpy()


# ── Position-split CV ──────────────────────────────────────────────────────────
print("\nRunning position-split 5-fold CV ...")
train_pos  = df_train['mutant'].apply(lambda x: int(x[1:-1])).values
unique_pos = np.unique(train_pos)
np.random.seed(42)
np.random.shuffle(unique_pos)
fold_size  = len(unique_pos) // 5
cv_scores  = []

for fold in range(5):
    val_pos  = set(unique_pos[fold*fold_size:(fold+1)*fold_size])
    val_mask = np.array([p in val_pos for p in train_pos])
    tr_mask  = ~val_mask

    gp_model, gp_lik = train_gp(X_train[tr_mask], y_train[tr_mask])
    mu, _            = gp_predict(gp_model, gp_lik, X_train[val_mask])
    r, _             = spearmanr(y_train[val_mask], mu)
    cv_scores.append(r)
    print(f"  Fold {fold+1}: {r:.4f}")

print(f"\nGP CV Spearman: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# ── Train final GP on all data ─────────────────────────────────────────────────
print("\nTraining final GP on all data ...")
final_gp, final_lik   = train_gp(X_train, y_train)
mu_test, std_test     = gp_predict(final_gp, final_lik, X_test)
y_pred                = np.clip(mu_test, 0, 1)

# ── Save submission ────────────────────────────────────────────────────────────
submission = pd.DataFrame({'id': range(len(df_test)), 'DMS_score': y_pred})
submission.to_csv(f'{DATA_DIR}/predictions.csv', index=False)
submission.to_csv(f'{DATA_DIR}/submission_round{ROUND}.csv', index=False)
print(f"\nSaved predictions.csv - upload to Kaggle")

# ── Top 10 predicted mutations ─────────────────────────────────────────────────
df_test_out               = df_test.copy()
df_test_out['fitness']    = y_pred
df_test_out['uncertainty'] = std_test
top10 = df_test_out.nlargest(10, 'fitness')[['mutant', 'fitness', 'uncertainty']]
print("\nTop 10 predicted high-fitness mutations:")
print(top10.to_string(index=False))

Train: 1240 | Test: 11324 | Seq: 656
Loading ESM-2 cache ...
  656 positions loaded
Extracting features ...

Running position-split 5-fold CV ...
  Iter 50/300 | Loss: -0.3429 | Lengthscales: [1.938 3.398 2.917 2.29 ]
  Iter 100/300 | Loss: -0.3585 | Lengthscales: [1.642 4.299 2.743 1.782]
  Iter 150/300 | Loss: -0.3585 | Lengthscales: [1.614 4.942 2.473 1.836]
  Iter 200/300 | Loss: -0.3595 | Lengthscales: [1.572 5.51  2.389 1.923]
  Iter 250/300 | Loss: -0.3569 | Lengthscales: [1.608 5.919 2.377 1.841]
  Iter 300/300 | Loss: -0.3545 | Lengthscales: [1.614 6.109 2.434 1.833]
  Fold 1: 0.5595
  Iter 50/300 | Loss: -0.2997 | Lengthscales: [2.038 2.978 3.    2.5  ]
  Iter 100/300 | Loss: -0.3230 | Lengthscales: [1.663 3.443 2.977 1.813]
  Iter 150/300 | Loss: -0.3129 | Lengthscales: [1.545 4.013 3.183 1.654]
  Iter 200/300 | Loss: -0.3240 | Lengthscales: [1.613 4.586 3.291 1.743]
  Iter 250/300 | Loss: -0.3200 | Lengthscales: [1.627 5.055 3.315 1.675]
  Iter 300/300 | Loss: -0.3188 | Len

gradescope leaderboard submission

In [4]:
import pandas as pd, os
DATA_DIR = os.path.expanduser('~/v_files/GT/sem2/MLB/hackathon/Hackathon_data')

# 1. Fix predictions.csv format
df_test = pd.read_csv(f'{DATA_DIR}/test.csv')
preds   = pd.read_csv(f'{DATA_DIR}/predictions.csv')

submission = pd.DataFrame({
    'mutant':            df_test['mutant'].values,
    'DMS_score_predicted': preds['DMS_score'].values
})
submission.to_csv(f'{DATA_DIR}/predictions.csv', index=False)
print("predictions.csv saved")
print(submission.head(3))

# 2. top10.txt from GP results
top10_mutants = [
    'E513W', 'E513Q', 'Y505W', 'N546C', 'E513M',
    'Y505M', 'E513K', 'Q484C', 'Q554C', 'N488M'
]

# Verify all are in test set and not in train set
df_train = pd.read_csv(f'{DATA_DIR}/train.csv')
train_mutants = set(df_train['mutant'].values)
test_mutants  = set(df_test['mutant'].values)

for m in top10_mutants:
    if m not in test_mutants:
        print(f"WARNING: {m} not in test set")
    if m in train_mutants:
        print(f"WARNING: {m} is in training set")

with open(f'{DATA_DIR}/top10.txt', 'w') as f:
    f.write('\n'.join(top10_mutants))
print("\ntop10.txt saved")
print('\n'.join(top10_mutants))

predictions.csv saved
  mutant  DMS_score_predicted
0    V1D             0.541475
1    V1Y             0.427454
2    V1C             0.435380

top10.txt saved
E513W
E513Q
Y505W
N546C
E513M
Y505M
E513K
Q484C
Q554C
N488M


# Round 2 - Exploration

queries locations with high entropy/uncertainty (low ESM-2 confidence)

### Error Analysis:

In [10]:
import pandas as pd
import numpy as np
import os

DATA_DIR = os.path.expanduser('~/v_files/GT/sem2/MLB/hackathon/Hackathon_data')

# Load fresh with query data
df_analysis = pd.read_csv(f'{DATA_DIR}/train.csv')
df_q1       = pd.read_csv(f'{DATA_DIR}/query1_labeled.csv')
df_analysis = pd.concat([df_analysis, df_q1], ignore_index=True)
df_analysis = df_analysis.drop_duplicates(subset='mutant').reset_index(drop=True)

print(f"Total training samples: {len(df_analysis)}")  # should be 1240

df_analysis['pos']   = df_analysis['mutant'].apply(lambda x: int(x[1:-1]))
df_analysis['mt_aa'] = df_analysis['mutant'].apply(lambda x: x[-1])
df_analysis['wt_aa'] = df_analysis['mutant'].apply(lambda x: x[0])

# GP predictions on training data
mu_train, _ = gp_predict(final_gp, final_lik, X_train)
df_analysis['gp_pred'] = mu_train
df_analysis['error']   = abs(df_analysis['gp_pred'] - df_analysis['DMS_score'])

# Per-position error
pos_error = df_analysis.groupby('pos')['error'].mean().sort_values(ascending=False)
print("\nTop 10 highest error positions:")
print(pos_error.head(10))
print("\nBottom 10 lowest error positions:")
print(pos_error.tail(10))

# Per mutation type error
mt_error = df_analysis.groupby('mt_aa')['error'].mean().sort_values(ascending=False)
print("\nError by mutant AA type:")
print(mt_error)

# Per WT AA error
wt_error = df_analysis.groupby('wt_aa')['error'].mean().sort_values(ascending=False)
print("\nError by WT AA type:")
print(wt_error.head(10))

Total training samples: 1240

Top 10 highest error positions:
pos
558    0.436444
226    0.382424
544    0.294432
393    0.286116
44     0.277572
366    0.264742
188    0.246598
328    0.245516
387    0.225977
611    0.217255
Name: error, dtype: float64

Bottom 10 lowest error positions:
pos
349    0.013762
358    0.012420
533    0.011997
437    0.007890
543    0.007878
545    0.005332
552    0.002920
603    0.001981
260    0.000828
616    0.000476
Name: error, dtype: float64

Error by mutant AA type:
mt_aa
F    0.155099
A    0.142782
L    0.141514
H    0.137326
W    0.135988
I    0.135122
D    0.134097
P    0.128158
V    0.126261
S    0.123680
M    0.120260
C    0.116265
T    0.114482
Q    0.112956
Y    0.111540
G    0.108283
K    0.106542
R    0.102040
N    0.096743
E    0.094884
Name: error, dtype: float64

Error by WT AA type:
wt_aa
T    0.193442
R    0.145863
S    0.141881
L    0.137461
C    0.137158
A    0.135849
V    0.133325
P    0.132027
D    0.131100
I    0.128374
Name: error

/Applications/miniconda3/lib/python3.10/site-packages/gpytorch/models/exact_gp.py:284: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


Positions: 558, 226, 544, 393, 44 have very high error — these are positions where the GP is most wrong. Querying more mutations at or near these positions would help most.

Error analysis revealed the GP struggled most at positions 558, 226, and 544, motivating the addition of BLOSUM62, physicochemical, and VHSE features to better capture substitution-level properties that ESM-2 alone does not encode.

### using GP + BLOSUM (for physiochemical features)

In [14]:
"""
Protein Fitness Prediction — Gaussian Process + Maximum Entropy Query
======================================================================
Input features (all from ESM-2 cache, generalize to unseen positions):
  1. zero_shot_score   = log P(mut) - log P(wt)
  2. position_entropy  = Shannon entropy of ESM distribution at position
  3. wt_log_prob       = log P(wt | context)
  4. normalized_pos    = position / seq_len  (structural context proxy)

Kernel: Matern-5/2 with ARD

Query strategy (Round 2): Maximum Entropy
  - Select 100 unseen positions with highest ESM-2 positional entropy
  - Pure exploration: queries positions where ESM-2 is most uncertain
  - No model needed for query selection — purely from cache
  - Goal: maximize position coverage so GP has more anchor points

"""

import os
import torch
import gpytorch
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler

# ── Config ─────────────────────────────────────────────────────────────────────
DATA_DIR   = os.path.expanduser('~/v_files/GT/sem2/MLB/hackathon/Hackathon_data')
TRAIN_PATH = f'{DATA_DIR}/train.csv'
TEST_PATH  = f'{DATA_DIR}/test.csv'
FASTA_PATH = f'{DATA_DIR}/sequence.fasta'
CACHE_PATH = f'{DATA_DIR}/esm2_150M_zeroshot_scores.npz'

QUERY_PATHS = [f'{DATA_DIR}/query1_labeled.csv']  # ← update each round
ROUND       = 2                                    # ← update each round
GP_ITERS    = 300
N_QUERY     = 100

# ── Load data ──────────────────────────────────────────────────────────────────
with open(FASTA_PATH) as f:
    sequence_wt = f.readlines()[1].strip()
SEQ_LEN = len(sequence_wt)

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)
for qp in QUERY_PATHS:
    df_train = pd.concat([df_train, pd.read_csv(qp)], ignore_index=True)
df_train = df_train.drop_duplicates(subset='mutant').reset_index(drop=True)

print(f"Train: {len(df_train)} | Test: {len(df_test)} | Seq: {SEQ_LEN}")

# ── Load ESM-2 cache ───────────────────────────────────────────────────────────
AAs = list('ACDEFGHIKLMNPQRSTVWY')
print("Loading ESM-2 cache ...")
pos_logprobs = np.load(CACHE_PATH, allow_pickle=True)['pos_logprobs'].item()
print(f"  {len(pos_logprobs)} positions loaded")

# ── Feature extraction ─────────────────────────────────────────────────────────
BLOSUM62 = {('A','A'):4,('A','R'):-1,('A','N'):-2,('A','D'):-2,('A','C'):0,('A','Q'):-1,('A','E'):-1,('A','G'):0,('A','H'):-2,('A','I'):-1,('A','L'):-1,('A','K'):-1,('A','M'):-1,('A','F'):-2,('A','P'):-1,('A','S'):1,('A','T'):0,('A','W'):-3,('A','Y'):-2,('A','V'):0,('R','A'):-1,('R','R'):5,('R','N'):0,('R','D'):-2,('R','C'):-3,('R','Q'):1,('R','E'):0,('R','G'):-2,('R','H'):0,('R','I'):-3,('R','L'):-2,('R','K'):2,('R','M'):-1,('R','F'):-3,('R','P'):-2,('R','S'):-1,('R','T'):-1,('R','W'):-3,('R','Y'):-2,('R','V'):-3,('N','A'):-2,('N','R'):0,('N','N'):6,('N','D'):1,('N','C'):-3,('N','Q'):0,('N','E'):0,('N','G'):0,('N','H'):1,('N','I'):-3,('N','L'):-3,('N','K'):0,('N','M'):-2,('N','F'):-3,('N','P'):-2,('N','S'):1,('N','T'):0,('N','W'):-4,('N','Y'):-2,('N','V'):-3,('D','A'):-2,('D','R'):-2,('D','N'):1,('D','D'):6,('D','C'):-3,('D','Q'):0,('D','E'):2,('D','G'):-1,('D','H'):-1,('D','I'):-3,('D','L'):-4,('D','K'):-1,('D','M'):-3,('D','F'):-3,('D','P'):-1,('D','S'):0,('D','T'):-1,('D','W'):-4,('D','Y'):-3,('D','V'):-3,('C','A'):0,('C','R'):-3,('C','N'):-3,('C','D'):-3,('C','C'):9,('C','Q'):-3,('C','E'):-4,('C','G'):-3,('C','H'):-3,('C','I'):-1,('C','L'):-1,('C','K'):-3,('C','M'):-1,('C','F'):-2,('C','P'):-3,('C','S'):-1,('C','T'):-1,('C','W'):-2,('C','Y'):-2,('C','V'):-1,('Q','A'):-1,('Q','R'):1,('Q','N'):0,('Q','D'):0,('Q','C'):-3,('Q','Q'):5,('Q','E'):2,('Q','G'):-2,('Q','H'):0,('Q','I'):-3,('Q','L'):-2,('Q','K'):1,('Q','M'):0,('Q','F'):-3,('Q','P'):-1,('Q','S'):0,('Q','T'):-1,('Q','W'):-2,('Q','Y'):-1,('Q','V'):-2,('E','A'):-1,('E','R'):0,('E','N'):0,('E','D'):2,('E','C'):-4,('E','Q'):2,('E','E'):5,('E','G'):-2,('E','H'):0,('E','I'):-3,('E','L'):-3,('E','K'):1,('E','M'):-2,('E','F'):-3,('E','P'):-1,('E','S'):0,('E','T'):-1,('E','W'):-3,('E','Y'):-2,('E','V'):-2,('G','A'):0,('G','R'):-2,('G','N'):0,('G','D'):-1,('G','C'):-3,('G','Q'):-2,('G','E'):-2,('G','G'):6,('G','H'):-2,('G','I'):-4,('G','L'):-4,('G','K'):-2,('G','M'):-3,('G','F'):-3,('G','P'):-2,('G','S'):0,('G','T'):-2,('G','W'):-2,('G','Y'):-3,('G','V'):-3,('H','A'):-2,('H','R'):0,('H','N'):1,('H','D'):-1,('H','C'):-3,('H','Q'):0,('H','E'):0,('H','G'):-2,('H','H'):8,('H','I'):-3,('H','L'):-3,('H','K'):-1,('H','M'):-2,('H','F'):-1,('H','P'):-2,('H','S'):-1,('H','T'):-2,('H','W'):-2,('H','Y'):2,('H','V'):-3,('I','A'):-1,('I','R'):-3,('I','N'):-3,('I','D'):-3,('I','C'):-1,('I','Q'):-3,('I','E'):-3,('I','G'):-4,('I','H'):-3,('I','I'):4,('I','L'):2,('I','K'):-3,('I','M'):1,('I','F'):0,('I','P'):-3,('I','S'):-2,('I','T'):-1,('I','W'):-3,('I','Y'):-1,('I','V'):3,('L','A'):-1,('L','R'):-2,('L','N'):-3,('L','D'):-4,('L','C'):-1,('L','Q'):-2,('L','E'):-3,('L','G'):-4,('L','H'):-3,('L','I'):2,('L','L'):4,('L','K'):-2,('L','M'):2,('L','F'):0,('L','P'):-3,('L','S'):-2,('L','T'):-1,('L','W'):-2,('L','Y'):-1,('L','V'):1,('K','A'):-1,('K','R'):2,('K','N'):0,('K','D'):-1,('K','C'):-3,('K','Q'):1,('K','E'):1,('K','G'):-2,('K','H'):-1,('K','I'):-3,('K','L'):-2,('K','K'):5,('K','M'):-1,('K','F'):-3,('K','P'):-1,('K','S'):0,('K','T'):-1,('K','W'):-3,('K','Y'):-2,('K','V'):-2,('M','A'):-1,('M','R'):-1,('M','N'):-2,('M','D'):-3,('M','C'):-1,('M','Q'):0,('M','E'):-2,('M','G'):-3,('M','H'):-2,('M','I'):1,('M','L'):2,('M','K'):-1,('M','M'):5,('M','F'):0,('M','P'):-2,('M','S'):-1,('M','T'):-1,('M','W'):-1,('M','Y'):-1,('M','V'):1,('F','A'):-2,('F','R'):-3,('F','N'):-3,('F','D'):-3,('F','C'):-2,('F','Q'):-3,('F','E'):-3,('F','G'):-3,('F','H'):-1,('F','I'):0,('F','L'):0,('F','K'):-3,('F','M'):0,('F','F'):6,('F','P'):-4,('F','S'):-2,('F','T'):-2,('F','W'):1,('F','Y'):3,('F','V'):-1,('P','A'):-1,('P','R'):-2,('P','N'):-2,('P','D'):-1,('P','C'):-3,('P','Q'):-1,('P','E'):-1,('P','G'):-2,('P','H'):-2,('P','I'):-3,('P','L'):-3,('P','K'):-1,('P','M'):-2,('P','F'):-4,('P','P'):7,('P','S'):-1,('P','T'):-1,('P','W'):-4,('P','Y'):-3,('P','V'):-2,('S','A'):1,('S','R'):-1,('S','N'):1,('S','D'):0,('S','C'):-1,('S','Q'):0,('S','E'):0,('S','G'):0,('S','H'):-1,('S','I'):-2,('S','L'):-2,('S','K'):0,('S','M'):-1,('S','F'):-2,('S','P'):-1,('S','S'):4,('S','T'):1,('S','W'):-3,('S','Y'):-2,('S','V'):-2,('T','A'):0,('T','R'):-1,('T','N'):0,('T','D'):-1,('T','C'):-1,('T','Q'):-1,('T','E'):-1,('T','G'):-2,('T','H'):-2,('T','I'):-1,('T','L'):-1,('T','K'):-1,('T','M'):-1,('T','F'):-2,('T','P'):-1,('T','S'):1,('T','T'):5,('T','W'):-2,('T','Y'):-2,('T','V'):0,('W','A'):-3,('W','R'):-3,('W','N'):-4,('W','D'):-4,('W','C'):-2,('W','Q'):-2,('W','E'):-3,('W','G'):-2,('W','H'):-2,('W','I'):-3,('W','L'):-2,('W','K'):-3,('W','M'):-1,('W','F'):1,('W','P'):-4,('W','S'):-3,('W','T'):-2,('W','W'):11,('W','Y'):2,('W','V'):-3,('Y','A'):-2,('Y','R'):-2,('Y','N'):-2,('Y','D'):-3,('Y','C'):-2,('Y','Q'):-1,('Y','E'):-2,('Y','G'):-3,('Y','H'):2,('Y','I'):-1,('Y','L'):-1,('Y','K'):-2,('Y','M'):-1,('Y','F'):3,('Y','P'):-3,('Y','S'):-2,('Y','T'):-2,('Y','W'):2,('Y','Y'):7,('Y','V'):-1,('V','A'):0,('V','R'):-3,('V','N'):-3,('V','D'):-3,('V','C'):-1,('V','Q'):-2,('V','E'):-2,('V','G'):-3,('V','H'):-3,('V','I'):3,('V','L'):1,('V','K'):-2,('V','M'):1,('V','F'):-1,('V','P'):-2,('V','S'):-2,('V','T'):0,('V','W'):-3,('V','Y'):-1,('V','V'):4}
HYDROPHOBICITY = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
CHARGE  = {'A':0,'R':1,'N':0,'D':-1,'C':0,'Q':0,'E':-1,'G':0,'H':0.1,'I':0,'L':0,'K':1,'M':0,'F':0,'P':0,'S':0,'T':0,'W':0,'Y':0,'V':0}
VOLUME  = {'A':67,'R':148,'N':96,'D':91,'C':86,'Q':114,'E':109,'G':48,'H':118,'I':124,'L':124,'K':135,'M':124,'F':135,'P':90,'S':73,'T':93,'W':163,'Y':141,'V':105}

# VHSE: Vectors of Hydrophobic, Steric, and Electronic properties
# 8 PCA-derived descriptors per AA — more comprehensive than individual features
# Source: Mei et al. (2005), J. Chem. Inf. Model.
VHSE = {
    'A': [ 0.15,-1.11,-1.35,-0.92, 0.02,-0.91, 0.36,-0.48],
    'R': [-1.47, 1.45, 1.24, 1.27, 1.55, 1.47, 1.30, 0.83],
    'N': [-0.99, 0.00,-0.37, 0.69,-0.55, 0.85, 0.73,-0.80],
    'D': [-1.15, 0.67,-0.41,-0.01,-2.68, 1.31, 0.03, 0.56],
    'C': [ 0.18,-1.67,-0.46,-0.21, 0.00, 1.20,-1.61,-0.19],
    'Q': [-0.96, 0.12, 0.18, 0.16, 0.09, 0.42,-0.20,-0.41],
    'E': [-1.18, 0.40, 0.10, 0.36,-2.16,-0.17, 0.91, 0.02],
    'G': [-0.20,-1.53,-2.63, 2.28,-0.53,-1.18, 2.01,-1.34],
    'H': [-0.43,-0.25, 0.37, 0.19, 0.51, 1.28, 0.93, 0.65],
    'I': [ 1.27,-0.14, 0.30,-1.80, 0.30,-1.61,-0.16,-0.13],
    'L': [ 1.36, 0.07, 0.26,-0.80, 0.22,-1.37, 0.08,-0.62],
    'K': [-1.17, 0.70, 0.70, 0.80, 1.64, 0.67, 1.63, 0.13],
    'M': [ 1.01,-0.53, 0.43, 0.00, 0.23,-0.10,-0.86,-0.68],
    'F': [ 1.52, 0.61, 0.96,-0.16, 0.25, 0.28,-1.33,-0.20],
    'P': [ 0.22,-0.17,-0.50, 0.05,-0.01,-1.34,-0.19, 3.56],
    'S': [-0.67,-0.86,-1.07,-0.41,-0.32, 0.27,-0.64, 0.11],
    'T': [-0.34,-0.51,-0.55,-1.06, 0.01,-0.01,-0.79, 0.39],
    'W': [ 1.50, 2.06, 1.79, 0.75, 0.75,-0.13,-1.06,-0.46],
    'Y': [ 0.61, 1.60, 1.17, 0.73, 0.53, 0.25,-0.96,-0.52],
    'V': [ 0.76,-0.92,-0.17,-1.91, 0.22,-1.40,-0.24,-0.03],
}

def build_features(df):
    """
    16 features total:
      ESM-2 derived (4):
        1. zero_shot_score
        2. position_entropy
        3. wt_log_prob
        4. normalized_pos
      Physicochemical (4):
        5. BLOSUM62 score
        6. delta_hydrophobicity
        7. delta_charge
        8. delta_volume
      VHSE delta (8):
        9-16. delta VHSE[1-8] (mutant - wt)
               captures hydrophobic, steric, electronic property changes
    """
    feats = []
    for _, row in df.iterrows():
        m      = row['mutant']
        wt, mt = m[0], m[-1]
        pos    = int(m[1:-1])
        lp     = pos_logprobs[pos]

        # ESM-2 features
        zs       = lp[mt] - lp[wt]
        logprobs = np.array([lp[aa] for aa in AAs])
        probs    = np.exp(logprobs); probs /= probs.sum()
        entropy  = float(-np.sum(probs * np.log(probs + 1e-10)))
        wt_logp  = lp[wt]
        norm_pos = pos / SEQ_LEN

        # Physicochemical features
        blosum       = BLOSUM62.get((wt, mt), 0)
        delta_hydro  = HYDROPHOBICITY[mt] - HYDROPHOBICITY[wt]
        delta_charge = CHARGE[mt] - CHARGE[wt]
        delta_volume = VOLUME[mt] - VOLUME[wt]

        # VHSE delta (8 dimensions)
        delta_vhse = [VHSE[mt][i] - VHSE[wt][i] for i in range(8)]

        feats.append([zs, entropy, wt_logp, norm_pos,
                      blosum, delta_hydro, delta_charge, delta_volume,
                      *delta_vhse])
    return np.array(feats, dtype=np.float32)

print("Extracting features ...")
X_train_raw = build_features(df_train)
X_test_raw  = build_features(df_test)
y_train     = df_train['DMS_score'].values.astype(np.float32)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

# ── GP Model ───────────────────────────────────────────────────────────────────
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module  = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(
                nu=2.5,
                ard_num_dims=train_x.shape[1]
            )
        )

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x),
            self.covar_module(x)
        )


def train_gp(X, y, n_iter=GP_ITERS, lr=0.1):
    train_x = torch.tensor(X)
    train_y = torch.tensor(y)

    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model      = ExactGPModel(train_x, train_y, likelihood)
    model.train(); likelihood.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(n_iter):
        optimizer.zero_grad()
        loss = -mll(model(train_x), train_y)
        loss.backward()
        optimizer.step()
        if (i + 1) % 50 == 0:
            ls = model.covar_module.base_kernel.lengthscale.detach().numpy().flatten()
            print(f"  Iter {i+1}/{n_iter} | Loss: {loss.item():.4f} | "
                  f"Lengthscales: {np.round(ls, 3)}")

    return model, likelihood


def gp_predict(model, likelihood, X):
    model.eval(); likelihood.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred = likelihood(model(torch.tensor(X)))
    return pred.mean.numpy(), pred.stddev.numpy()


# ── Position-split CV ──────────────────────────────────────────────────────────
print("\nRunning position-split 5-fold CV ...")
train_pos  = df_train['mutant'].apply(lambda x: int(x[1:-1])).values
unique_pos = np.unique(train_pos)
np.random.seed(42)
np.random.shuffle(unique_pos)
fold_size  = len(unique_pos) // 5
cv_scores  = []

for fold in range(5):
    val_pos  = set(unique_pos[fold*fold_size:(fold+1)*fold_size])
    val_mask = np.array([p in val_pos for p in train_pos])
    tr_mask  = ~val_mask

    gp_model, gp_lik = train_gp(X_train[tr_mask], y_train[tr_mask])
    mu, _            = gp_predict(gp_model, gp_lik, X_train[val_mask])
    r, _             = spearmanr(y_train[val_mask], mu)
    cv_scores.append(r)
    print(f"  Fold {fold+1}: {r:.4f}")

print(f"\nGP CV Spearman: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# ── Train final GP on all data ─────────────────────────────────────────────────
print("\nTraining final GP on all data ...")
final_gp, final_lik = train_gp(X_train, y_train)
mu_test, std_test   = gp_predict(final_gp, final_lik, X_test)
y_pred              = np.clip(mu_test, 0, 1)

# ── Save submission ────────────────────────────────────────────────────────────
submission = pd.DataFrame({'id': range(len(df_test)), 'DMS_score': y_pred})
submission.to_csv(f'{DATA_DIR}/predictions.csv', index=False)
submission.to_csv(f'{DATA_DIR}/submission_round{ROUND}.csv', index=False)
print(f"\nSaved predictions.csv → upload to Kaggle")

# ── Top 10 predicted mutations ─────────────────────────────────────────────────
df_test_out                = df_test.copy()
df_test_out['fitness']     = y_pred
df_test_out['uncertainty'] = std_test
top10 = df_test_out.nlargest(10, 'fitness')[['mutant', 'fitness', 'uncertainty']]
print("\nTop 10 predicted high-fitness mutations:")
print(top10.to_string(index=False))

# ── Maximum Entropy Query Selection (Round 2 — Pure Exploration) ───────────────
"""
Query the 100 unseen positions with highest ESM-2 positional entropy.
entropy = -sum(P(aa) * log P(aa)) over all 20 AAs at that position.

High entropy → ESM-2 is uncertain which AA belongs here → zero-shot scores
are least reliable → labeled data gives maximum information to the GP.

Within each position, pick the mutation with the highest zero-shot score
for biological relevance.
"""

# Compute per-position entropy from cache
pos_entropy = {}
for pos, lp in pos_logprobs.items():
    logprobs = np.array([lp[aa] for aa in AAs])
    probs    = np.exp(logprobs); probs /= probs.sum()
    pos_entropy[pos] = float(-np.sum(probs * np.log(probs + 1e-10)))

# Exclude positions already in training data
seen_positions = set(df_train['mutant'].apply(lambda x: int(x[1:-1])))
train_mutants  = set(df_train['mutant'].values)  # for final safety check

df_q                   = df_test.copy()
df_q['pos']            = df_q['mutant'].apply(lambda x: int(x[1:-1]))
df_q['entropy']        = df_q['pos'].map(pos_entropy)
df_q['zs_score']       = X_test_raw[:, 0]
df_q['gp_fitness']     = y_pred
df_q['gp_uncertainty'] = std_test

query_df = (
    df_q[~df_q['pos'].isin(seen_positions)]
    .sort_values(['entropy', 'zs_score'], ascending=[False, False])
    .drop_duplicates(subset='pos')
    .head(N_QUERY)
)

# Safety check: ensure no training mutants included
leaked = [m for m in query_df['mutant'] if m in train_mutants]
if leaked:
    print(f"WARNING: {len(leaked)} training mutants detected — removing")
    query_df = query_df[~query_df['mutant'].isin(train_mutants)]

assert len(query_df) <= 100, "Query exceeds 100 mutants"
assert query_df['pos'].nunique() == len(query_df), "Duplicate positions in query"

# Save query.txt (Gradescope format: one mutant per line, no header)
with open(f'{DATA_DIR}/query_round{ROUND}.txt', 'w') as f:
    f.write('\n'.join(query_df['mutant'].tolist()))

query_df[['mutant', 'entropy', 'gp_fitness', 'gp_uncertainty']].to_csv(
    f'{DATA_DIR}/query_round{ROUND}.csv', index=False
)

print(f"\nMaximum Entropy Query — Round {ROUND}")
print(f"  Positions queried: {query_df['pos'].nunique()}")
print(f"  Entropy range: {query_df['entropy'].min():.3f} – {query_df['entropy'].max():.3f}")
print(f"  Training mutant leakage: {len(leaked)} (should be 0)")
print(f"\nTop 10 queried mutations (highest entropy):")
print(query_df[['mutant', 'entropy', 'gp_fitness', 'gp_uncertainty']].head(10).to_string(index=False))
print(f"\nSaved query_round{ROUND}.txt → submit to Gradescope")

print(f"""
── Round {ROUND} complete ──────────────────────────────────────────────────────
  predictions.csv          → Kaggle upload
  query_round{ROUND}.txt          → Gradescope hackathon API

After receiving labels:
  1. Save as {DATA_DIR}/query{ROUND}_labeled.csv  (columns: mutant, DMS_score)
  2. Add to QUERY_PATHS
  3. Set ROUND = {ROUND + 1}
  4. Switch to predict_gp_ucb.py for round 3 (GP-UCB query)
─────────────────────────────────────────────────────────────────────────────
""")

Train: 1240 | Test: 11324 | Seq: 656
Loading ESM-2 cache ...
  656 positions loaded
Extracting features ...

Running position-split 5-fold CV ...
  Iter 50/300 | Loss: -0.3011 | Lengthscales: [1.974 3.417 2.568 2.546 4.213 4.515 5.16  4.216 4.533 4.399 4.469 4.479
 4.997 4.573 4.59  4.463]
  Iter 100/300 | Loss: -0.3464 | Lengthscales: [1.101 3.278 2.043 1.392 4.932 5.954 7.301 5.399 6.253 5.431 5.923 6.154
 6.855 6.117 6.117 5.968]
  Iter 150/300 | Loss: -0.3481 | Lengthscales: [1.133 3.54  2.103 1.367 5.363 6.917 8.514 6.45  7.386 6.255 7.113 7.267
 7.944 7.18  7.064 7.083]
  Iter 200/300 | Loss: -0.3528 | Lengthscales: [1.142 3.771 2.055 1.383 5.76  7.736 9.618 7.298 8.384 6.993 8.069 8.233
 8.978 8.167 7.962 7.989]
  Iter 250/300 | Loss: -0.3580 | Lengthscales: [ 1.213  4.038  2.183  1.401  6.047  8.435 10.566  8.149  9.206  7.804
  9.007  9.063  9.846  9.062  8.763  8.984]
  Iter 300/300 | Loss: -0.3591 | Lengthscales: [ 1.208  4.126  2.067  1.458  6.224  8.992 11.406  8.929  9.90

# Round 3 - UCB

using Upper Confidence Bound (UCB = mean + k * std) acquision function for querying. Balances exploration (high uncertainty) and exploitation (high predicted fitness).

The k parameter controls the tradeoff:

k=0 → pure exploitation (just sort by mean, ignore uncertainty)

k=1 → balanced (our choice)

k=2+ → heavy exploration (prioritize uncertain regions even if predicted fitness is low)

drop_duplicates(subset='pos') keeps only the first occurrence of each position after sorting by ucb_score descending.

This enforces position diversity — instead of querying 100 mutations at 5 highly-ranked positions, you query 1 mutation at each of 100 different positions. That maximizes information gain since each new position gives the GP a completely new anchor point.

In [24]:
"""
Protein Fitness Prediction — Gaussian Process + Maximum Entropy Query
======================================================================
Input features (all from ESM-2 cache, generalize to unseen positions):
  1. zero_shot_score   = log P(mut) - log P(wt)
  2. position_entropy  = Shannon entropy of ESM distribution at position
  3. wt_log_prob       = log P(wt | context)
  4. normalized_pos    = position / seq_len  (structural context proxy)

Kernel: Matern-5/2 with ARD

Query strategy Round 3 : GP-UCB
  - Select mutations with highest Upper Confidence Bound (UCB = mean + β * std)
  - Balances exploration (high uncertainty) and exploitation (high predicted fitness)
"""

import os
import torch
import gpytorch
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler

# ── Config ─────────────────────────────────────────────────────────────────────
DATA_DIR   = os.path.expanduser('~/v_files/GT/sem2/MLB/hackathon/Hackathon_data')
TRAIN_PATH = f'{DATA_DIR}/train.csv'
TEST_PATH  = f'{DATA_DIR}/test.csv'
FASTA_PATH = f'{DATA_DIR}/sequence.fasta'
CACHE_PATH = f'{DATA_DIR}/esm2_150M_zeroshot_scores.npz'

QUERY_PATHS = [f'{DATA_DIR}/query1_labeled.csv',
               f'{DATA_DIR}/query2_labeled.csv',
               f'{DATA_DIR}/query3_labeled.csv'] # <-- update with each round's labeled queries
ROUND       = 4  # ← update each round
GP_ITERS    = 500
N_QUERY     = 100

# ── Load data ──────────────────────────────────────────────────────────────────
with open(FASTA_PATH) as f:
    sequence_wt = f.readlines()[1].strip()
SEQ_LEN = len(sequence_wt)

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)
for qp in QUERY_PATHS:
    df_train = pd.concat([df_train, pd.read_csv(qp)], ignore_index=True)
df_train = df_train.drop_duplicates(subset='mutant').reset_index(drop=True)

print(f"Train: {len(df_train)} | Test: {len(df_test)} | Seq: {SEQ_LEN}")

# ── Load ESM-2 cache ───────────────────────────────────────────────────────────
AAs = list('ACDEFGHIKLMNPQRSTVWY')
print("Loading ESM-2 cache ...")
pos_logprobs = np.load(CACHE_PATH, allow_pickle=True)['pos_logprobs'].item()
print(f"  {len(pos_logprobs)} positions loaded")

# ── Feature extraction ─────────────────────────────────────────────────────────
BLOSUM62 = {('A','A'):4,('A','R'):-1,('A','N'):-2,('A','D'):-2,('A','C'):0,('A','Q'):-1,('A','E'):-1,('A','G'):0,('A','H'):-2,('A','I'):-1,('A','L'):-1,('A','K'):-1,('A','M'):-1,('A','F'):-2,('A','P'):-1,('A','S'):1,('A','T'):0,('A','W'):-3,('A','Y'):-2,('A','V'):0,('R','A'):-1,('R','R'):5,('R','N'):0,('R','D'):-2,('R','C'):-3,('R','Q'):1,('R','E'):0,('R','G'):-2,('R','H'):0,('R','I'):-3,('R','L'):-2,('R','K'):2,('R','M'):-1,('R','F'):-3,('R','P'):-2,('R','S'):-1,('R','T'):-1,('R','W'):-3,('R','Y'):-2,('R','V'):-3,('N','A'):-2,('N','R'):0,('N','N'):6,('N','D'):1,('N','C'):-3,('N','Q'):0,('N','E'):0,('N','G'):0,('N','H'):1,('N','I'):-3,('N','L'):-3,('N','K'):0,('N','M'):-2,('N','F'):-3,('N','P'):-2,('N','S'):1,('N','T'):0,('N','W'):-4,('N','Y'):-2,('N','V'):-3,('D','A'):-2,('D','R'):-2,('D','N'):1,('D','D'):6,('D','C'):-3,('D','Q'):0,('D','E'):2,('D','G'):-1,('D','H'):-1,('D','I'):-3,('D','L'):-4,('D','K'):-1,('D','M'):-3,('D','F'):-3,('D','P'):-1,('D','S'):0,('D','T'):-1,('D','W'):-4,('D','Y'):-3,('D','V'):-3,('C','A'):0,('C','R'):-3,('C','N'):-3,('C','D'):-3,('C','C'):9,('C','Q'):-3,('C','E'):-4,('C','G'):-3,('C','H'):-3,('C','I'):-1,('C','L'):-1,('C','K'):-3,('C','M'):-1,('C','F'):-2,('C','P'):-3,('C','S'):-1,('C','T'):-1,('C','W'):-2,('C','Y'):-2,('C','V'):-1,('Q','A'):-1,('Q','R'):1,('Q','N'):0,('Q','D'):0,('Q','C'):-3,('Q','Q'):5,('Q','E'):2,('Q','G'):-2,('Q','H'):0,('Q','I'):-3,('Q','L'):-2,('Q','K'):1,('Q','M'):0,('Q','F'):-3,('Q','P'):-1,('Q','S'):0,('Q','T'):-1,('Q','W'):-2,('Q','Y'):-1,('Q','V'):-2,('E','A'):-1,('E','R'):0,('E','N'):0,('E','D'):2,('E','C'):-4,('E','Q'):2,('E','E'):5,('E','G'):-2,('E','H'):0,('E','I'):-3,('E','L'):-3,('E','K'):1,('E','M'):-2,('E','F'):-3,('E','P'):-1,('E','S'):0,('E','T'):-1,('E','W'):-3,('E','Y'):-2,('E','V'):-2,('G','A'):0,('G','R'):-2,('G','N'):0,('G','D'):-1,('G','C'):-3,('G','Q'):-2,('G','E'):-2,('G','G'):6,('G','H'):-2,('G','I'):-4,('G','L'):-4,('G','K'):-2,('G','M'):-3,('G','F'):-3,('G','P'):-2,('G','S'):0,('G','T'):-2,('G','W'):-2,('G','Y'):-3,('G','V'):-3,('H','A'):-2,('H','R'):0,('H','N'):1,('H','D'):-1,('H','C'):-3,('H','Q'):0,('H','E'):0,('H','G'):-2,('H','H'):8,('H','I'):-3,('H','L'):-3,('H','K'):-1,('H','M'):-2,('H','F'):-1,('H','P'):-2,('H','S'):-1,('H','T'):-2,('H','W'):-2,('H','Y'):2,('H','V'):-3,('I','A'):-1,('I','R'):-3,('I','N'):-3,('I','D'):-3,('I','C'):-1,('I','Q'):-3,('I','E'):-3,('I','G'):-4,('I','H'):-3,('I','I'):4,('I','L'):2,('I','K'):-3,('I','M'):1,('I','F'):0,('I','P'):-3,('I','S'):-2,('I','T'):-1,('I','W'):-3,('I','Y'):-1,('I','V'):3,('L','A'):-1,('L','R'):-2,('L','N'):-3,('L','D'):-4,('L','C'):-1,('L','Q'):-2,('L','E'):-3,('L','G'):-4,('L','H'):-3,('L','I'):2,('L','L'):4,('L','K'):-2,('L','M'):2,('L','F'):0,('L','P'):-3,('L','S'):-2,('L','T'):-1,('L','W'):-2,('L','Y'):-1,('L','V'):1,('K','A'):-1,('K','R'):2,('K','N'):0,('K','D'):-1,('K','C'):-3,('K','Q'):1,('K','E'):1,('K','G'):-2,('K','H'):-1,('K','I'):-3,('K','L'):-2,('K','K'):5,('K','M'):-1,('K','F'):-3,('K','P'):-1,('K','S'):0,('K','T'):-1,('K','W'):-3,('K','Y'):-2,('K','V'):-2,('M','A'):-1,('M','R'):-1,('M','N'):-2,('M','D'):-3,('M','C'):-1,('M','Q'):0,('M','E'):-2,('M','G'):-3,('M','H'):-2,('M','I'):1,('M','L'):2,('M','K'):-1,('M','M'):5,('M','F'):0,('M','P'):-2,('M','S'):-1,('M','T'):-1,('M','W'):-1,('M','Y'):-1,('M','V'):1,('F','A'):-2,('F','R'):-3,('F','N'):-3,('F','D'):-3,('F','C'):-2,('F','Q'):-3,('F','E'):-3,('F','G'):-3,('F','H'):-1,('F','I'):0,('F','L'):0,('F','K'):-3,('F','M'):0,('F','F'):6,('F','P'):-4,('F','S'):-2,('F','T'):-2,('F','W'):1,('F','Y'):3,('F','V'):-1,('P','A'):-1,('P','R'):-2,('P','N'):-2,('P','D'):-1,('P','C'):-3,('P','Q'):-1,('P','E'):-1,('P','G'):-2,('P','H'):-2,('P','I'):-3,('P','L'):-3,('P','K'):-1,('P','M'):-2,('P','F'):-4,('P','P'):7,('P','S'):-1,('P','T'):-1,('P','W'):-4,('P','Y'):-3,('P','V'):-2,('S','A'):1,('S','R'):-1,('S','N'):1,('S','D'):0,('S','C'):-1,('S','Q'):0,('S','E'):0,('S','G'):0,('S','H'):-1,('S','I'):-2,('S','L'):-2,('S','K'):0,('S','M'):-1,('S','F'):-2,('S','P'):-1,('S','S'):4,('S','T'):1,('S','W'):-3,('S','Y'):-2,('S','V'):-2,('T','A'):0,('T','R'):-1,('T','N'):0,('T','D'):-1,('T','C'):-1,('T','Q'):-1,('T','E'):-1,('T','G'):-2,('T','H'):-2,('T','I'):-1,('T','L'):-1,('T','K'):-1,('T','M'):-1,('T','F'):-2,('T','P'):-1,('T','S'):1,('T','T'):5,('T','W'):-2,('T','Y'):-2,('T','V'):0,('W','A'):-3,('W','R'):-3,('W','N'):-4,('W','D'):-4,('W','C'):-2,('W','Q'):-2,('W','E'):-3,('W','G'):-2,('W','H'):-2,('W','I'):-3,('W','L'):-2,('W','K'):-3,('W','M'):-1,('W','F'):1,('W','P'):-4,('W','S'):-3,('W','T'):-2,('W','W'):11,('W','Y'):2,('W','V'):-3,('Y','A'):-2,('Y','R'):-2,('Y','N'):-2,('Y','D'):-3,('Y','C'):-2,('Y','Q'):-1,('Y','E'):-2,('Y','G'):-3,('Y','H'):2,('Y','I'):-1,('Y','L'):-1,('Y','K'):-2,('Y','M'):-1,('Y','F'):3,('Y','P'):-3,('Y','S'):-2,('Y','T'):-2,('Y','W'):2,('Y','Y'):7,('Y','V'):-1,('V','A'):0,('V','R'):-3,('V','N'):-3,('V','D'):-3,('V','C'):-1,('V','Q'):-2,('V','E'):-2,('V','G'):-3,('V','H'):-3,('V','I'):3,('V','L'):1,('V','K'):-2,('V','M'):1,('V','F'):-1,('V','P'):-2,('V','S'):-2,('V','T'):0,('V','W'):-3,('V','Y'):-1,('V','V'):4}
HYDROPHOBICITY = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
CHARGE  = {'A':0,'R':1,'N':0,'D':-1,'C':0,'Q':0,'E':-1,'G':0,'H':0.1,'I':0,'L':0,'K':1,'M':0,'F':0,'P':0,'S':0,'T':0,'W':0,'Y':0,'V':0}
VOLUME  = {'A':67,'R':148,'N':96,'D':91,'C':86,'Q':114,'E':109,'G':48,'H':118,'I':124,'L':124,'K':135,'M':124,'F':135,'P':90,'S':73,'T':93,'W':163,'Y':141,'V':105}

# VHSE: Vectors of Hydrophobic, Steric, and Electronic properties
# 8 PCA-derived descriptors per AA — more comprehensive than individual features
# Source: Mei et al. (2005), J. Chem. Inf. Model.
VHSE = {
    'A': [ 0.15,-1.11,-1.35,-0.92, 0.02,-0.91, 0.36,-0.48],
    'R': [-1.47, 1.45, 1.24, 1.27, 1.55, 1.47, 1.30, 0.83],
    'N': [-0.99, 0.00,-0.37, 0.69,-0.55, 0.85, 0.73,-0.80],
    'D': [-1.15, 0.67,-0.41,-0.01,-2.68, 1.31, 0.03, 0.56],
    'C': [ 0.18,-1.67,-0.46,-0.21, 0.00, 1.20,-1.61,-0.19],
    'Q': [-0.96, 0.12, 0.18, 0.16, 0.09, 0.42,-0.20,-0.41],
    'E': [-1.18, 0.40, 0.10, 0.36,-2.16,-0.17, 0.91, 0.02],
    'G': [-0.20,-1.53,-2.63, 2.28,-0.53,-1.18, 2.01,-1.34],
    'H': [-0.43,-0.25, 0.37, 0.19, 0.51, 1.28, 0.93, 0.65],
    'I': [ 1.27,-0.14, 0.30,-1.80, 0.30,-1.61,-0.16,-0.13],
    'L': [ 1.36, 0.07, 0.26,-0.80, 0.22,-1.37, 0.08,-0.62],
    'K': [-1.17, 0.70, 0.70, 0.80, 1.64, 0.67, 1.63, 0.13],
    'M': [ 1.01,-0.53, 0.43, 0.00, 0.23,-0.10,-0.86,-0.68],
    'F': [ 1.52, 0.61, 0.96,-0.16, 0.25, 0.28,-1.33,-0.20],
    'P': [ 0.22,-0.17,-0.50, 0.05,-0.01,-1.34,-0.19, 3.56],
    'S': [-0.67,-0.86,-1.07,-0.41,-0.32, 0.27,-0.64, 0.11],
    'T': [-0.34,-0.51,-0.55,-1.06, 0.01,-0.01,-0.79, 0.39],
    'W': [ 1.50, 2.06, 1.79, 0.75, 0.75,-0.13,-1.06,-0.46],
    'Y': [ 0.61, 1.60, 1.17, 0.73, 0.53, 0.25,-0.96,-0.52],
    'V': [ 0.76,-0.92,-0.17,-1.91, 0.22,-1.40,-0.24,-0.03],
}

def build_features(df):
    """
    16 features total:
      ESM-2 derived (4):
        1. zero_shot_score
        2. position_entropy
        3. wt_log_prob
        4. normalized_pos
      Physicochemical (4):
        5. BLOSUM62 score
        6. delta_hydrophobicity
        7. delta_charge
        8. delta_volume
      VHSE delta (8):
        9-16. delta VHSE[1-8] (mutant - wt)
               captures hydrophobic, steric, electronic property changes
    """
    feats = []
    for _, row in df.iterrows():
        m      = row['mutant']
        wt, mt = m[0], m[-1]
        pos    = int(m[1:-1])
        lp     = pos_logprobs[pos]

        # ESM-2 features
        zs       = lp[mt] - lp[wt]
        logprobs = np.array([lp[aa] for aa in AAs])
        probs    = np.exp(logprobs); probs /= probs.sum()
        entropy  = float(-np.sum(probs * np.log(probs + 1e-10)))
        wt_logp  = lp[wt]
        norm_pos = pos / SEQ_LEN

        # Physicochemical features
        blosum       = BLOSUM62.get((wt, mt), 0)
        delta_hydro  = HYDROPHOBICITY[mt] - HYDROPHOBICITY[wt]
        delta_charge = CHARGE[mt] - CHARGE[wt]
        delta_volume = VOLUME[mt] - VOLUME[wt]

        # VHSE delta (8 dimensions)
        delta_vhse = [VHSE[mt][i] - VHSE[wt][i] for i in range(8)]

        feats.append([zs, entropy, wt_logp, norm_pos,
                      blosum, delta_hydro, delta_charge, delta_volume,
                      *delta_vhse])
    return np.array(feats, dtype=np.float32)

print("Extracting features ...")
X_train_raw = build_features(df_train)
X_test_raw  = build_features(df_test)
y_train     = df_train['DMS_score'].values.astype(np.float32)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

# ── GP Model ───────────────────────────────────────────────────────────────────
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module  = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(
                nu=2.5,
                ard_num_dims=train_x.shape[1]
            )
        )

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x),
            self.covar_module(x)
        )


def train_gp(X, y, n_iter=GP_ITERS, lr=0.1):
    train_x = torch.tensor(X)
    train_y = torch.tensor(y)

    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model      = ExactGPModel(train_x, train_y, likelihood)
    model.train(); likelihood.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(n_iter):
        optimizer.zero_grad()
        loss = -mll(model(train_x), train_y)
        loss.backward()
        optimizer.step()
        if (i + 1) % 50 == 0:
            ls = model.covar_module.base_kernel.lengthscale.detach().numpy().flatten()
            print(f"  Iter {i+1}/{n_iter} | Loss: {loss.item():.4f} | "
                  f"Lengthscales: {np.round(ls, 3)}")

    return model, likelihood


def gp_predict(model, likelihood, X):
    model.eval(); likelihood.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred = likelihood(model(torch.tensor(X)))
    return pred.mean.numpy(), pred.stddev.numpy()


# ── Position-split CV ──────────────────────────────────────────────────────────
print("\nRunning position-split 5-fold CV ...")
train_pos  = df_train['mutant'].apply(lambda x: int(x[1:-1])).values
unique_pos = np.unique(train_pos)
np.random.seed(42)
np.random.shuffle(unique_pos)
fold_size  = len(unique_pos) // 5
cv_scores  = []

for fold in range(5):
    val_pos  = set(unique_pos[fold*fold_size:(fold+1)*fold_size])
    val_mask = np.array([p in val_pos for p in train_pos])
    tr_mask  = ~val_mask

    gp_model, gp_lik = train_gp(X_train[tr_mask], y_train[tr_mask])
    mu, _            = gp_predict(gp_model, gp_lik, X_train[val_mask])
    r, _             = spearmanr(y_train[val_mask], mu)
    cv_scores.append(r)
    print(f"  Fold {fold+1}: {r:.4f}")

print(f"\nGP CV Spearman: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# ── Train final GP on all data ─────────────────────────────────────────────────
print("\nTraining final GP on all data ...")
final_gp, final_lik = train_gp(X_train, y_train)
mu_test, std_test   = gp_predict(final_gp, final_lik, X_test)
y_pred              = np.clip(mu_test, 0, 1)

# ── Save submission ────────────────────────────────────────────────────────────
submission = pd.DataFrame({'id': range(len(df_test)), 'DMS_score': y_pred})
submission.to_csv(f'{DATA_DIR}/predictions_3.csv', index=False)
submission.to_csv(f'{DATA_DIR}/submission_round{ROUND}.csv', index=False)
print(f"\nSaved predictions_3.csv → upload to Kaggle")

# ── Top 10 predicted mutations ─────────────────────────────────────────────────
df_test_out                = df_test.copy()
df_test_out['fitness']     = y_pred
df_test_out['uncertainty'] = std_test
top10 = df_test_out.nlargest(10, 'fitness')[['mutant', 'fitness', 'uncertainty']]
print("\nTop 10 predicted high-fitness mutations:")
print(top10.to_string(index=False))

# ── UCB Query Selection (Round 3) ───────────────

K_UCB = 1.0

seen_positions = set(df_train['mutant'].apply(lambda x: int(x[1:-1])))
train_mutants  = set(df_train['mutant'].values)

df_q                   = df_test.copy()
df_q['pos']            = df_q['mutant'].apply(lambda x: int(x[1:-1]))
df_q['gp_fitness']     = y_pred
df_q['gp_uncertainty'] = std_test
df_q['ucb_score']      = df_q['gp_fitness'] + K_UCB * df_q['gp_uncertainty']

query_df = (
    df_q[~df_q['pos'].isin(seen_positions)]
    .sort_values('ucb_score', ascending=False)
    .drop_duplicates(subset='pos')
    .head(N_QUERY)
)

leaked = [m for m in query_df['mutant'] if m in train_mutants]
if leaked:
    print(f"WARNING: {len(leaked)} training mutants — removing")
    query_df = query_df[~query_df['mutant'].isin(train_mutants)]

assert len(query_df) <= 100
# with open(f'{DATA_DIR}/query_round{ROUND}.txt', 'w') as f:
#     f.write('\n'.join(query_df['mutant'].tolist()))
# query_df[['mutant','ucb_score','gp_fitness','gp_uncertainty']].to_csv(
#     f'{DATA_DIR}/query_round{ROUND}.csv', index=False)
# print(f"\nGP-UCB Query — Round {ROUND} (k={K_UCB})")
# print(f"  Positions: {query_df['pos'].nunique()}")
# print(f"  Leakage: {len(leaked)} (should be 0)")
# print(query_df[['mutant','ucb_score','gp_fitness','gp_uncertainty']].head(10).to_string(index=False))
# print(f"\nSaved query_round{ROUND}.txt → submit to Gradescope")

Train: 1440 | Test: 11324 | Seq: 656
Loading ESM-2 cache ...
  656 positions loaded
Extracting features ...

Running position-split 5-fold CV ...
  Iter 50/500 | Loss: -0.3147 | Lengthscales: [1.447 2.832 2.431 2.486 4.216 4.383 5.117 4.247 4.635 4.314 4.508 4.295
 4.92  4.479 4.586 4.761]
  Iter 100/500 | Loss: -0.3574 | Lengthscales: [1.042 2.574 2.263 1.017 5.317 5.66  7.349 5.389 6.201 5.397 6.039 5.664
 6.927 5.967 6.162 6.117]
  Iter 150/500 | Loss: -0.3596 | Lengthscales: [1.098 3.152 2.358 1.074 5.974 6.501 8.652 6.245 7.213 6.182 7.093 6.707
 8.13  7.045 7.262 7.092]
  Iter 200/500 | Loss: -0.3628 | Lengthscales: [1.136 3.496 2.276 1.057 6.566 7.138 9.744 6.87  8.063 6.909 7.946 7.606
 9.136 8.016 8.214 7.961]
  Iter 250/500 | Loss: -0.3648 | Lengthscales: [ 1.23   3.865  2.412  1.082  6.978  7.791 10.788  7.505  8.862  7.565
  8.706  8.486 10.073  8.834  9.023  8.665]
  Iter 300/500 | Loss: -0.3670 | Lengthscales: [ 1.174  3.998  2.393  1.103  7.261  8.321 11.644  8.109  9.57

### optimizing further

adding RBF kernel + ReduceLROnPlateau + increased iterations

In [28]:
"""
Protein Fitness Prediction — GP Optimized
==========================================
16 features (same as 0.prev submission):
  ESM-2 (4) + Physicochemical (4) + VHSE delta (8)

Optimizations:
  1. Composite kernel: Matern-5/2 + RBF (sum)
     - Matern captures rough local structure
     - RBF captures smooth global trends
  2. 750 iterations 

"""

import os
import torch
import gpytorch
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler

DATA_DIR   = os.path.expanduser('~/v_files/GT/sem2/MLB/hackathon/Hackathon_data')
TRAIN_PATH = f'{DATA_DIR}/train.csv'
TEST_PATH  = f'{DATA_DIR}/test.csv'
FASTA_PATH = f'{DATA_DIR}/sequence.fasta'
CACHE_PATH = f'{DATA_DIR}/esm2_150M_zeroshot_scores.npz'

QUERY_PATHS = [f'{DATA_DIR}/query1_labeled.csv',
               f'{DATA_DIR}/query2_labeled.csv',
               f'{DATA_DIR}/query3_labeled.csv']
ROUND       = 4
GP_ITERS    = 1500

with open(FASTA_PATH) as f:
    sequence_wt = f.readlines()[1].strip()
SEQ_LEN = len(sequence_wt)

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)
for qp in QUERY_PATHS:
    df_train = pd.concat([df_train, pd.read_csv(qp)], ignore_index=True)
df_train = df_train.drop_duplicates(subset='mutant').reset_index(drop=True)
print(f"Train: {len(df_train)} | Test: {len(df_test)} | Seq: {SEQ_LEN}")

AAs = list('ACDEFGHIKLMNPQRSTVWY')
print("Loading ESM-2 cache ...")
pos_logprobs = np.load(CACHE_PATH, allow_pickle=True)['pos_logprobs'].item()

BLOSUM62 = {('A','A'):4,('A','R'):-1,('A','N'):-2,('A','D'):-2,('A','C'):0,('A','Q'):-1,('A','E'):-1,('A','G'):0,('A','H'):-2,('A','I'):-1,('A','L'):-1,('A','K'):-1,('A','M'):-1,('A','F'):-2,('A','P'):-1,('A','S'):1,('A','T'):0,('A','W'):-3,('A','Y'):-2,('A','V'):0,('R','A'):-1,('R','R'):5,('R','N'):0,('R','D'):-2,('R','C'):-3,('R','Q'):1,('R','E'):0,('R','G'):-2,('R','H'):0,('R','I'):-3,('R','L'):-2,('R','K'):2,('R','M'):-1,('R','F'):-3,('R','P'):-2,('R','S'):-1,('R','T'):-1,('R','W'):-3,('R','Y'):-2,('R','V'):-3,('N','A'):-2,('N','R'):0,('N','N'):6,('N','D'):1,('N','C'):-3,('N','Q'):0,('N','E'):0,('N','G'):0,('N','H'):1,('N','I'):-3,('N','L'):-3,('N','K'):0,('N','M'):-2,('N','F'):-3,('N','P'):-2,('N','S'):1,('N','T'):0,('N','W'):-4,('N','Y'):-2,('N','V'):-3,('D','A'):-2,('D','R'):-2,('D','N'):1,('D','D'):6,('D','C'):-3,('D','Q'):0,('D','E'):2,('D','G'):-1,('D','H'):-1,('D','I'):-3,('D','L'):-4,('D','K'):-1,('D','M'):-3,('D','F'):-3,('D','P'):-1,('D','S'):0,('D','T'):-1,('D','W'):-4,('D','Y'):-3,('D','V'):-3,('C','A'):0,('C','R'):-3,('C','N'):-3,('C','D'):-3,('C','C'):9,('C','Q'):-3,('C','E'):-4,('C','G'):-3,('C','H'):-3,('C','I'):-1,('C','L'):-1,('C','K'):-3,('C','M'):-1,('C','F'):-2,('C','P'):-3,('C','S'):-1,('C','T'):-1,('C','W'):-2,('C','Y'):-2,('C','V'):-1,('Q','A'):-1,('Q','R'):1,('Q','N'):0,('Q','D'):0,('Q','C'):-3,('Q','Q'):5,('Q','E'):2,('Q','G'):-2,('Q','H'):0,('Q','I'):-3,('Q','L'):-2,('Q','K'):1,('Q','M'):0,('Q','F'):-3,('Q','P'):-1,('Q','S'):0,('Q','T'):-1,('Q','W'):-2,('Q','Y'):-1,('Q','V'):-2,('E','A'):-1,('E','R'):0,('E','N'):0,('E','D'):2,('E','C'):-4,('E','Q'):2,('E','E'):5,('E','G'):-2,('E','H'):0,('E','I'):-3,('E','L'):-3,('E','K'):1,('E','M'):-2,('E','F'):-3,('E','P'):-1,('E','S'):0,('E','T'):-1,('E','W'):-3,('E','Y'):-2,('E','V'):-2,('G','A'):0,('G','R'):-2,('G','N'):0,('G','D'):-1,('G','C'):-3,('G','Q'):-2,('G','E'):-2,('G','G'):6,('G','H'):-2,('G','I'):-4,('G','L'):-4,('G','K'):-2,('G','M'):-3,('G','F'):-3,('G','P'):-2,('G','S'):0,('G','T'):-2,('G','W'):-2,('G','Y'):-3,('G','V'):-3,('H','A'):-2,('H','R'):0,('H','N'):1,('H','D'):-1,('H','C'):-3,('H','Q'):0,('H','E'):0,('H','G'):-2,('H','H'):8,('H','I'):-3,('H','L'):-3,('H','K'):-1,('H','M'):-2,('H','F'):-1,('H','P'):-2,('H','S'):-1,('H','T'):-2,('H','W'):-2,('H','Y'):2,('H','V'):-3,('I','A'):-1,('I','R'):-3,('I','N'):-3,('I','D'):-3,('I','C'):-1,('I','Q'):-3,('I','E'):-3,('I','G'):-4,('I','H'):-3,('I','I'):4,('I','L'):2,('I','K'):-3,('I','M'):1,('I','F'):0,('I','P'):-3,('I','S'):-2,('I','T'):-1,('I','W'):-3,('I','Y'):-1,('I','V'):3,('L','A'):-1,('L','R'):-2,('L','N'):-3,('L','D'):-4,('L','C'):-1,('L','Q'):-2,('L','E'):-3,('L','G'):-4,('L','H'):-3,('L','I'):2,('L','L'):4,('L','K'):-2,('L','M'):2,('L','F'):0,('L','P'):-3,('L','S'):-2,('L','T'):-1,('L','W'):-2,('L','Y'):-1,('L','V'):1,('K','A'):-1,('K','R'):2,('K','N'):0,('K','D'):-1,('K','C'):-3,('K','Q'):1,('K','E'):1,('K','G'):-2,('K','H'):-1,('K','I'):-3,('K','L'):-2,('K','K'):5,('K','M'):-1,('K','F'):-3,('K','P'):-1,('K','S'):0,('K','T'):-1,('K','W'):-3,('K','Y'):-2,('K','V'):-2,('M','A'):-1,('M','R'):-1,('M','N'):-2,('M','D'):-3,('M','C'):-1,('M','Q'):0,('M','E'):-2,('M','G'):-3,('M','H'):-2,('M','I'):1,('M','L'):2,('M','K'):-1,('M','M'):5,('M','F'):0,('M','P'):-2,('M','S'):-1,('M','T'):-1,('M','W'):-1,('M','Y'):-1,('M','V'):1,('F','A'):-2,('F','R'):-3,('F','N'):-3,('F','D'):-3,('F','C'):-2,('F','Q'):-3,('F','E'):-3,('F','G'):-3,('F','H'):-1,('F','I'):0,('F','L'):0,('F','K'):-3,('F','M'):0,('F','F'):6,('F','P'):-4,('F','S'):-2,('F','T'):-2,('F','W'):1,('F','Y'):3,('F','V'):-1,('P','A'):-1,('P','R'):-2,('P','N'):-2,('P','D'):-1,('P','C'):-3,('P','Q'):-1,('P','E'):-1,('P','G'):-2,('P','H'):-2,('P','I'):-3,('P','L'):-3,('P','K'):-1,('P','M'):-2,('P','F'):-4,('P','P'):7,('P','S'):-1,('P','T'):-1,('P','W'):-4,('P','Y'):-3,('P','V'):-2,('S','A'):1,('S','R'):-1,('S','N'):1,('S','D'):0,('S','C'):-1,('S','Q'):0,('S','E'):0,('S','G'):0,('S','H'):-1,('S','I'):-2,('S','L'):-2,('S','K'):0,('S','M'):-1,('S','F'):-2,('S','P'):-1,('S','S'):4,('S','T'):1,('S','W'):-3,('S','Y'):-2,('S','V'):-2,('T','A'):0,('T','R'):-1,('T','N'):0,('T','D'):-1,('T','C'):-1,('T','Q'):-1,('T','E'):-1,('T','G'):-2,('T','H'):-2,('T','I'):-1,('T','L'):-1,('T','K'):-1,('T','M'):-1,('T','F'):-2,('T','P'):-1,('T','S'):1,('T','T'):5,('T','W'):-2,('T','Y'):-2,('T','V'):0,('W','A'):-3,('W','R'):-3,('W','N'):-4,('W','D'):-4,('W','C'):-2,('W','Q'):-2,('W','E'):-3,('W','G'):-2,('W','H'):-2,('W','I'):-3,('W','L'):-2,('W','K'):-3,('W','M'):-1,('W','F'):1,('W','P'):-4,('W','S'):-3,('W','T'):-2,('W','W'):11,('W','Y'):2,('W','V'):-3,('Y','A'):-2,('Y','R'):-2,('Y','N'):-2,('Y','D'):-3,('Y','C'):-2,('Y','Q'):-1,('Y','E'):-2,('Y','G'):-3,('Y','H'):2,('Y','I'):-1,('Y','L'):-1,('Y','K'):-2,('Y','M'):-1,('Y','F'):3,('Y','P'):-3,('Y','S'):-2,('Y','T'):-2,('Y','W'):2,('Y','Y'):7,('Y','V'):-1,('V','A'):0,('V','R'):-3,('V','N'):-3,('V','D'):-3,('V','C'):-1,('V','Q'):-2,('V','E'):-2,('V','G'):-3,('V','H'):-3,('V','I'):3,('V','L'):1,('V','K'):-2,('V','M'):1,('V','F'):-1,('V','P'):-2,('V','S'):-2,('V','T'):0,('V','W'):-3,('V','Y'):-1,('V','V'):4}
HYDROPHOBICITY = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
CHARGE  = {'A':0,'R':1,'N':0,'D':-1,'C':0,'Q':0,'E':-1,'G':0,'H':0.1,'I':0,'L':0,'K':1,'M':0,'F':0,'P':0,'S':0,'T':0,'W':0,'Y':0,'V':0}
VOLUME  = {'A':67,'R':148,'N':96,'D':91,'C':86,'Q':114,'E':109,'G':48,'H':118,'I':124,'L':124,'K':135,'M':124,'F':135,'P':90,'S':73,'T':93,'W':163,'Y':141,'V':105}
VHSE = {'A':[ 0.15,-1.11,-1.35,-0.92, 0.02,-0.91, 0.36,-0.48],'R':[-1.47, 1.45, 1.24, 1.27, 1.55, 1.47, 1.30, 0.83],'N':[-0.99, 0.00,-0.37, 0.69,-0.55, 0.85, 0.73,-0.80],'D':[-1.15, 0.67,-0.41,-0.01,-2.68, 1.31, 0.03, 0.56],'C':[ 0.18,-1.67,-0.46,-0.21, 0.00, 1.20,-1.61,-0.19],'Q':[-0.96, 0.12, 0.18, 0.16, 0.09, 0.42,-0.20,-0.41],'E':[-1.18, 0.40, 0.10, 0.36,-2.16,-0.17, 0.91, 0.02],'G':[-0.20,-1.53,-2.63, 2.28,-0.53,-1.18, 2.01,-1.34],'H':[-0.43,-0.25, 0.37, 0.19, 0.51, 1.28, 0.93, 0.65],'I':[ 1.27,-0.14, 0.30,-1.80, 0.30,-1.61,-0.16,-0.13],'L':[ 1.36, 0.07, 0.26,-0.80, 0.22,-1.37, 0.08,-0.62],'K':[-1.17, 0.70, 0.70, 0.80, 1.64, 0.67, 1.63, 0.13],'M':[ 1.01,-0.53, 0.43, 0.00, 0.23,-0.10,-0.86,-0.68],'F':[ 1.52, 0.61, 0.96,-0.16, 0.25, 0.28,-1.33,-0.20],'P':[ 0.22,-0.17,-0.50, 0.05,-0.01,-1.34,-0.19, 3.56],'S':[-0.67,-0.86,-1.07,-0.41,-0.32, 0.27,-0.64, 0.11],'T':[-0.34,-0.51,-0.55,-1.06, 0.01,-0.01,-0.79, 0.39],'W':[ 1.50, 2.06, 1.79, 0.75, 0.75,-0.13,-1.06,-0.46],'Y':[ 0.61, 1.60, 1.17, 0.73, 0.53, 0.25,-0.96,-0.52],'V':[ 0.76,-0.92,-0.17,-1.91, 0.22,-1.40,-0.24,-0.03]}

def build_features(df):
    feats = []
    for _, row in df.iterrows():
        m      = row['mutant']
        wt, mt = m[0], m[-1]
        pos    = int(m[1:-1])
        lp     = pos_logprobs[pos]
        zs       = lp[mt] - lp[wt]
        logprobs = np.array([lp[aa] for aa in AAs])
        probs    = np.exp(logprobs); probs /= probs.sum()
        entropy  = float(-np.sum(probs * np.log(probs + 1e-10)))
        wt_logp  = lp[wt]
        norm_pos = pos / SEQ_LEN
        blosum       = BLOSUM62.get((wt, mt), 0)
        delta_hydro  = HYDROPHOBICITY[mt] - HYDROPHOBICITY[wt]
        delta_charge = CHARGE[mt] - CHARGE[wt]
        delta_volume = VOLUME[mt] - VOLUME[wt]
        delta_vhse   = [VHSE[mt][i] - VHSE[wt][i] for i in range(8)]
        feats.append([zs, entropy, wt_logp, norm_pos,
                      blosum, delta_hydro, delta_charge, delta_volume,
                      *delta_vhse])
    return np.array(feats, dtype=np.float32)

print("Extracting features ...")
X_train_raw = build_features(df_train)
X_test_raw  = build_features(df_test)
y_train     = df_train['DMS_score'].values.astype(np.float32)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

# ── GP Model with composite kernel: Matern + RBF ───────────────────────────────
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module  = gpytorch.means.ConstantMean()
        ndim = train_x.shape[1]
        # Composite kernel: Matern-5/2 (rough structure) + RBF (smooth trends)
        # Both with ARD — independent lengthscale per feature
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(nu=2.5, ard_num_dims=ndim)
        ) + gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=ndim)
        )

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x), self.covar_module(x)
        )


def train_gp(X, y, n_iter=GP_ITERS, lr=0.05, verbose=True):
    """ReduceLROnPlateau: halves LR when loss stalls, down to min 1e-4."""
    train_x = torch.tensor(X); train_y = torch.tensor(y)
    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model = ExactGPModel(train_x, train_y, likelihood)
    model.train(); likelihood.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=30, min_lr=1e-4
    )
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
    for i in range(n_iter):
        optimizer.zero_grad()
        loss = -mll(model(train_x), train_y)
        loss.backward(); optimizer.step()
        scheduler.step(loss.item())
        if verbose and (i+1) % 150 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Iter {i+1}/{n_iter} | Loss: {loss.item():.4f} | LR: {current_lr:.5f}")
    return model, likelihood


def gp_predict(model, likelihood, X):
    model.eval(); likelihood.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred = likelihood(model(torch.tensor(X)))
    return pred.mean.numpy(), pred.stddev.numpy()


# ── Position-split CV ──────────────────────────────────────────────────────────
print("\nRunning position-split 5-fold CV ...")
train_pos  = df_train['mutant'].apply(lambda x: int(x[1:-1])).values
unique_pos = np.unique(train_pos)
np.random.seed(42); np.random.shuffle(unique_pos)
fold_size  = len(unique_pos) // 5
cv_scores  = []

for fold in range(5):
    val_pos  = set(unique_pos[fold*fold_size:(fold+1)*fold_size])
    val_mask = np.array([p in val_pos for p in train_pos])
    tr_mask  = ~val_mask
    gp_model, gp_lik = train_gp(X_train[tr_mask], y_train[tr_mask], verbose=False)
    mu, _            = gp_predict(gp_model, gp_lik, X_train[val_mask])
    r, _             = spearmanr(y_train[val_mask], mu)
    cv_scores.append(r)
    print(f"  Fold {fold+1}: {r:.4f}")

print(f"\nGP CV Spearman: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# ── Train final GP ─────────────────────────────────────────────────────────────
print("\nTraining final GP on all data ...")
final_gp, final_lik = train_gp(X_train, y_train, verbose=True)
mu_test, std_test   = gp_predict(final_gp, final_lik, X_test)
y_pred              = np.clip(mu_test, 0, 1)

# ── Save ───────────────────────────────────────────────────────────────────────
submission = pd.DataFrame({'id': range(len(df_test)), 'DMS_score': y_pred})
submission.to_csv(f'{DATA_DIR}/predictions.csv', index=False)
submission.to_csv(f'{DATA_DIR}/submission_round{ROUND}_opt.csv', index=False)
print(f"\nSaved predictions.csv → upload to Kaggle")

df_out = df_test.copy()
df_out['fitness']     = y_pred
df_out['uncertainty'] = std_test
top10 = df_out.nlargest(10, 'fitness')[['mutant', 'fitness', 'uncertainty']]
print("\nTop 10 predicted high-fitness mutations:")
print(top10.to_string(index=False))

# ── top10.txt ──────────────────────────────────────────────────────────────────
train_mutants = set(df_train['mutant'].values)
test_mutants  = set(df_test['mutant'].values)
top10_mutants = top10['mutant'].tolist()

for m in top10_mutants:
    if m not in test_mutants:
        print(f"WARNING: {m} not in test set")
    if m in train_mutants:
        print(f"WARNING: {m} in training set — should not be recommended")

with open(f'{DATA_DIR}/top10.txt', 'w') as f:
    f.write('\n'.join(top10_mutants))
print(f"\nSaved top10.txt")

# ── Gradescope format: predictions.csv ────────────────────────────────────────
gradescope = pd.DataFrame({
    'mutant':              df_test['mutant'].values,
    'DMS_score_predicted': y_pred
})
gradescope.to_csv(f'{DATA_DIR}/predictions.csv', index=False)
print("Reformatted predictions.csv for Gradescope (mutant, DMS_score_predicted)")
print(gradescope.head(3).to_string(index=False))

Train: 1440 | Test: 11324 | Seq: 656
Loading ESM-2 cache ...
Extracting features ...

Running position-split 5-fold CV ...
  Fold 1: 0.6541
  Fold 2: 0.7432
  Fold 3: 0.7473
  Fold 4: 0.7622
  Fold 5: 0.7473

GP CV Spearman: 0.7308 ± 0.0389

Training final GP on all data ...
  Iter 150/1500 | Loss: -0.3493 | LR: 0.05000
  Iter 300/1500 | Loss: -0.3669 | LR: 0.02500
  Iter 450/1500 | Loss: -0.3763 | LR: 0.00625
  Iter 600/1500 | Loss: -0.3747 | LR: 0.00020
  Iter 750/1500 | Loss: -0.3789 | LR: 0.00010
  Iter 900/1500 | Loss: -0.3771 | LR: 0.00010
  Iter 1050/1500 | Loss: -0.3779 | LR: 0.00010
  Iter 1200/1500 | Loss: -0.3791 | LR: 0.00010
  Iter 1350/1500 | Loss: -0.3784 | LR: 0.00010
  Iter 1500/1500 | Loss: -0.3748 | LR: 0.00010

Saved predictions.csv → upload to Kaggle

Top 10 predicted high-fitness mutations:
mutant  fitness  uncertainty
 S411L 0.952916     0.162088
 S411I 0.951464     0.162102
 S411V 0.950959     0.160389
 Q484Y 0.950660     0.159722
 S604L 0.948058     0.163802
 N